# Sanity Check - Step 03: Bad Channels Detect

Überprüft:
- Bad channels identifiziert
- QC-Reports erstellt
- Markierung in Raw-Objekten
- Statistiken plausibel

In [ ]:
import importlib
import os
import sys
from pathlib import Path
import mne

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from eeg_pipeline import config
except ModuleNotFoundError:
    PIPELINE_DIR = ROOT / 'eeg_pipeline'
    if not PIPELINE_DIR.exists():
        PIPELINE_DIR = ROOT.parent.parent / 'eeg_pipeline'
    if str(PIPELINE_DIR) not in sys.path:
        sys.path.insert(0, str(PIPELINE_DIR))
    config = importlib.import_module('config')

print('Setup erfolgreich')
print(f'Gefundene Subjects in config: {config.SUBJECTS}')

Setup erfolgreich


In [ ]:
# Manuelle Auswahl fuer diesen Notebook-Run
subject_id = "01"  # z.B. "02"
persons = ["P1", "P2"]  # oder nur ["P1"]
invalid = [p for p in persons if p not in {"P1", "P2"}]
if invalid:
    raise ValueError(f"Ungueltige Person(en): {invalid}. Erlaubt: P1,P2")
os.environ["EEG_SUBJECT"] = subject_id
os.environ["EEG_PERSONS"] = ",".join(persons)
print(f"Manuell gesetzt: EEG_SUBJECT={os.environ['EEG_SUBJECT']}, EEG_PERSONS={os.environ['EEG_PERSONS']}")

## 1. Bad Channels Detected Files laden

In [ ]:
subject_from_env = os.getenv('EEG_SUBJECT')
subject_default = globals().get('subject_id', subject_from_env or (config.SUBJECTS[0] if config.SUBJECTS else None))
if subject_default is None:
    raise ValueError('Kein Subject gefunden. Setze subject_id in der Zelle oder EEG_SUBJECT als Umgebungsvariable.')

subject_id = str(subject_default)
if subject_id.isdigit() and len(subject_id) == 1:
    subject_id = subject_id.zfill(2)

persons = globals().get('persons', ['P1', 'P2'])
print(f'Nutze subject_id={subject_id}, persons={persons}')

files = {}
for person in persons:
    path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    if path.exists():
        files[person] = mne.io.read_raw_fif(str(path), preload=False)
        print(f"✓ {person}: {path.name}")
    else:
        print(f"✗ {person}: File not found")

Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_32280\4267262547.py:7: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  files[person] = mne.io.read_raw_fif(str(path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
✓ P1: sub-01_P1_badchannels_detected.fif
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_32280\4267262547.py:7: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  files[person] = mne.io.read_raw_fif(str(path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
✓ P2: sub-01_P2_badchannels_detected.fif


## 2. Markierte Bad Channels

In [3]:
for person, raw in files.items():
    bads = raw.info.get('bads', [])
    
    print(f"\n=== {person} ===")
    print(f"Bad channels: {len(bads)}")
    if bads:
        print(f"  Kanäle: {', '.join(bads)}")
    else:
        print(f"  Keine Bad Channels markiert")


=== P1 ===
Bad channels: 0
  Keine Bad Channels markiert

=== P2 ===
Bad channels: 0
  Keine Bad Channels markiert


## 3. QC-Reports

In [ ]:
for person in persons:
    report_path = config.QC_DIR / f"sub-{subject_id}_{person}_bad_channels_detect.tsv"
    
    if report_path.exists():
        print(f"\n✓ {person}: {report_path.name}")
        with open(report_path, 'r') as f:
            lines = f.readlines()
            print(f"  Analyzeiert: {len(lines)-1} Kanäle")
            # Show first few lines
            print(f"  Header: {lines[0].strip()}")
            if len(lines) > 1:
                print(f"  Erste Zeile: {lines[1].strip()[:60]}...")
    else:
        print(f"\n✗ {person}: QC-Report nicht gefunden")


✓ P1: sub-01_P1_bad_channels_detect.tsv
  Analyzeiert: 64 Kanäle
  Header: subject_id	person	channel	std	robust_z	suggested	reason
  Erste Zeile: sub-01	P1	Fp1	7.356793351437e-04	-0.478710	no...

✓ P2: sub-01_P2_bad_channels_detect.tsv
  Analyzeiert: 64 Kanäle
  Header: subject_id	person	channel	std	robust_z	suggested	reason
  Erste Zeile: sub-01	P2	Fp1	1.487425417631e-03	0.398150	no...


## 4. Sanity Checks

In [5]:
for person, raw in files.items():
    print(f"\n{person}:")
    
    bads = raw.info.get('bads', [])
    total_eeg = len(mne.pick_types(raw.info, eeg=True))
    
    print(f"  Total EEG channels: {total_eeg}")
    print(f"  Bad channels: {len(bads)}")
    
    if len(bads) > total_eeg:
        print(f"  ✗ Fehler: Mehr Bad-Channels als EEG-Kanäle!")
    elif len(bads) == 0:
        print(f"  ✓ Keine Bad-Channels (oder alle passieren den Test)")
    else:
        print(f"  ✓ {len(bads)} Bad-Channels erkannt (plausibel)")


P1:
  Total EEG channels: 64
  Bad channels: 0
  ✓ Keine Bad-Channels (oder alle passieren den Test)

P2:
  Total EEG channels: 64
  Bad channels: 0
  ✓ Keine Bad-Channels (oder alle passieren den Test)
